## House Prices Prediction: Regression Techniques, A Starter Notebook

Welcome to the House Prices Prediction competition! This notebook aims to guide you through the process of predicting the final sales prices of homes in Ames, Iowa. With 79 explanatory variables describing various aspects of residential homes, this competition provides an excellent opportunity to practice and enhance your data science skills.

### Objectives:
- **Exploratory Data Analysis (EDA)**: Understand the dataset and uncover initial insights.
- **Data Cleaning and Preprocessing**: Handle missing values, encode categorical variables, and prepare the data for modeling.
- **Feature Engineering**: Create new features that can improve model performance.
- **Modeling**: Implement regression techniques.
- **Evaluation**: Assess model performance using Root Mean Squared Error (RMSE).

In [1]:
import pandas as pd
import numpy as np

#Ploating
import matplotlib.pyplot as plt
import seaborn as sns

#Emcoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

# x-y split
from sklearn.model_selection import train_test_split

#Scaling
from sklearn.preprocessing import StandardScaler

# Regression algorithms
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

#MSE AND MAE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from math import sqrt

ModuleNotFoundError: No module named 'seaborn'

# Exploratory Data Analysis

## Data Overview 

The dataset consists of **1,460 rows** and **81 columns**, each representing a residential home in Ames, Iowa. It is designed for **regression tasks**, with the goal of **predicting the final sale price (`SalePrice`) of homes** based on a wide variety of features describing the properties.

These features include:

* **Location and zoning** (`MSZoning`, `Neighborhood`, `LotFrontage`)
* **Property and lot characteristics** (`LotArea`, `Street`, `Alley`, `LotShape`)
* **House construction and condition** (`YearBuilt`, `BldgType`, `HouseStyle`, `OverallQual`, `OverallCond`)
* **Basement and garage details** (`BsmtQual`, `TotalBsmtSF`, `GarageType`, `GarageArea`)
* **Interior features** (`KitchenQual`, `TotRmsAbvGrd`, `Fireplaces`, `Functional`)
* **Amenities and external features** (`PoolArea`, `Fence`, `MiscFeature`, `WoodDeckSF`)
* **Sale details** (`MoSold`, `YrSold`, `SaleType`, `SaleCondition`)

In [ ]:
train = pd.read_csv("train.csv")
train.head()

In [ ]:
train.info()

In [ ]:
# Select only columns with int or float data types
numeric_columns = train.select_dtypes(include=['int64', 'float64'])

# Plot the describe function for numeric columns
numeric_columns.columns

In [ ]:
train.shape

In [ ]:
train.columns

## Data Quality Check

### Missing Data

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(train.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Data Heatmap', fontsize=16)
plt.xlabel('Columns')
plt.show()

In [ ]:
missing_values = train.isnull().sum() 
missing_values = (missing_values[missing_values > 0] / len(train) ) * 100
print(missing_values.sort_values(ascending=False))

### Handling Missing Values 

In [ ]:
train[train['PoolQC'].isna()]['PoolArea'].unique()

In [ ]:
train['PoolQC'].value_counts()

### The missing values are houses without pool, but too much outliers so we will drop the PoolArea and PoolQC

In [ ]:
train.drop(columns=['PoolQC','PoolArea'],inplace=True)

In [ ]:
train[train['FireplaceQu'].isna()]['Fireplaces'].unique()

### Util function for handling some missing data

In [ ]:
def plot_and_fill_nans(df, column_name, target_col='SalePrice', fill_value='None'):

    # Create figure with subplots
    plt.figure(figsize=(12, 6))
    
    # Before filling NaNs (original data)
    plt.subplot(1, 2, 1)
    sns.barplot(data=df, x=column_name, y=target_col, hue=column_name,
                estimator='mean', errorbar=None)
    plt.title(f'Original {column_name} (with NaNs)')
    plt.xticks(rotation=45)
    
    # Fill NaNs IN PLACE
    df[column_name].fillna(fill_value, inplace=True)
    
    # After filling NaNs
    plt.subplot(1, 2, 2)
    sns.barplot(data=df, x=column_name, y=target_col, hue=column_name,
                estimator='mean', errorbar=None)
    plt.title(f'After filling NaNs with "{fill_value}"')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_and_fill_nans(train, 'FireplaceQu')  

### We notice that there is a correlation between the fireplace quality and the saleprice as expected 

In [ ]:
plot_and_fill_nans(train, 'MiscFeature')  

In [ ]:
plot_and_fill_nans(train, 'Fence')  

In [ ]:
train['Street'].value_counts()

In [ ]:
train['Alley'].value_counts()

### We will drop the Street and Alley columns as there is no much variance and also too missing values

In [ ]:
train.drop(columns=['Street','Alley'],inplace=True)

In [ ]:
train['MasVnrType'].value_counts(dropna=False)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.boxplot(x='MasVnrType', y='SalePrice', data=train)
plt.xticks(rotation=45)
plt.title("SalePrice vs MasVnrType")
plt.show()

In [ ]:
train[train['MasVnrType'].isnull() & (train['MasVnrArea'] > 0)]

### We will impute with None or Unknown

In [ ]:
train.loc[(train['MasVnrType'].isnull()) & (train['MasVnrArea'] == 0), 'MasVnrType'] = 'None'
train['MasVnrType'] = train['MasVnrType'].fillna('Unknown')

### Other missing values

#### `LotFrontage` is a continuous numeric variable , highly correlated with Neighborhood and LotArea, Imputation by KNN or median by Neighborhood

In [ ]:
train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

In [ ]:
garage_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
train[garage_cols] = train[garage_cols].fillna('None')
train['GarageYrBlt'] = train['GarageYrBlt'].fillna(train['YearBuilt'])

In [ ]:
bsmt_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
train[bsmt_cols] = train[bsmt_cols].fillna('None')

In [ ]:
train['MasVnrArea'] = train['MasVnrArea'].fillna(0)

In [ ]:
train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(train.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Data Heatmap', fontsize=16)
plt.xlabel('Columns')
plt.show()

In [ ]:
train['Utilities'].value_counts()

In [ ]:
len(train[train['Condition1'] == train['Condition2']]) / len(train) * 100

In [ ]:
train['RoofMatl'].value_counts()

In [ ]:
len(train[train['BsmtFinType1'] == train['BsmtFinType2']]) / len(train) * 100

In [ ]:
train['LowQualFinSF'].value_counts()

In [ ]:
train['Functional'].value_counts()

#### Columns to Potentially Remove at First Glance

In [ ]:
cols_to_drop = [
    'Id',               # Unique identifier, not useful for modeling
    'Utilities',        # Very low variance (almost all values are 'AllPub')
    'Condition2',       # Often redundant with Condition1
    'RoofMatl',         # Highly imbalanced, dominated by a single category
    'BsmtFinType2',     # Often missing or redundant with BsmtFinType1
    'LowQualFinSF',     # Very rarely used
    'MiscVal',          # Mostly zero, minimal predictive power
    'GarageYrBlt',      # Strongly correlated with YearBuilt, usually identical
    'Functional',       # Most values are 'Typ' (Typical)
    'MoSold',           # Month of sale — little to no predictive signal
]

train.drop(columns=cols_to_drop, inplace=True)

print(f"{len(cols_to_drop)} columns dropped. Current dataset shape: {train.shape}")

### Duplicates Values

In [ ]:
duplicates_values = train.duplicated()
duplicates_values = (duplicates_values[duplicates_values > 0] / len(train) ) * 100
print(duplicates_values)

### Outliers

#### For the describe function, I will only plot meaningfull numerical columns because some numerical columns are just encoded categorical columns because those columns do not carry meaningful statistical interpretation unless the categories have a real ordinal relationship 
#### So I'll keep only the following columns
````
['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond',
       'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
       'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces',
       'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF',
       'EnclosedPorch', '3SsnPorch', 'ScreenPorch',
       'SalePrice']
````

In [ ]:
desc_df = train[['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond',
       'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
       'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces',
       'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF',
       'EnclosedPorch', '3SsnPorch', 'ScreenPorch',
       'SalePrice']].describe().T
desc_df

In [ ]:
# Plot as heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(desc_df[['count','mean', 'std', 'min', '25%', '50%', '75%', 'max']], 
            annot=True, fmt=".1f", cmap="YlGnBu")
plt.title('Heatmap of Summary Statistics for Numerical Features')
plt.tight_layout()
plt.show()

#### To better understand how to handle the variables, we will group them by category.

As an initial overview, we observe that most houses include **3 bedrooms**, **2 full bathrooms**, and a **2-car garage**. In contrast, rare houses feature less common additions such as **pools**, **three-season porches**, and **basements with high-end finishes**.
The **heatmap of the summary statistics above** offers further insights into missing numerical data.

<!-- Previously, we identified several variables with missing numerical values, including `LotFrontage`, `MasVnrArea`, and `GarageYrBlt`:

* **GarageYrBlt**: This variable is often redundant, as the garage is usually built the same year as the house. It can either be **dropped** or **filled using `YearBuilt`**.

* **LotFrontage**: Due to its **right-skewed distribution** and presence of outliers, this feature could be imputed using a **log transformation** or **K-Nearest Neighbors (KNN)** based on neighborhood-related variables.

* **MasVnrArea**: Since many values are missing and the absence likely indicates **no masonry veneer**, it is reasonable to **fill missing values with 0**. -->

We will also explore the following aspects in the next sections:

* #### Features with potential outliers
* #### Sparse features
* #### Ordinal / categorical-like numeric features
* #### Opportunities for feature engineering
* #### Features likely correlated with the sale price
* #### Right-skewed distributions

Each category will be examined and addressed in detail.

## Feature Engineering

### **1. Features with Potential Outliers**

Certain variables exhibit unusually high maximum values that deviate significantly from the rest of the distribution. These outliers may skew analysis or model training if not addressed. For example, **LotFrontage** has a maximum value of 313 ft, much higher than the 75th percentile (80 ft), and **LotArea** reaches 215,245 sq ft while most values are below 10,000. **MasVnrArea**, **TotalBsmtSF**, **GrLivArea**, **LowQualFinSF**, and **MiscVal** similarly show extreme maximums.
> 🟢 **Action**:  These should be visualized using boxplots or histograms, and handled via capping (e.g., winsorization) or log-transformation where appropriate.

In [ ]:
# Columns of interest
cols = ['LotFrontage', 'LotArea', 'MasVnrArea', 'TotalBsmtSF', 
        'GrLivArea']

# Set plot style
sns.set(style="whitegrid")

# Plot histograms
def plot_histograms(df, columns):
    plt.figure(figsize=(16, 12))
    for i, col in enumerate(columns):
        plt.subplot(4, 2, i + 1)
        sns.histplot(df[col].dropna(), bins=30, kde=True)
        plt.title(f'Distribution of {col}')
    plt.tight_layout()
    plt.suptitle("Histograms of Selected Features", fontsize=16, y=1.02)
    plt.show()

# Plot boxplots
def plot_boxplots(df, columns):
    plt.figure(figsize=(16, 12))
    for i, col in enumerate(columns):
        plt.subplot(4, 2, i + 1)
        sns.boxplot(x=df[col])
        plt.title(f'Boxplot of {col}')
    plt.tight_layout()
    plt.suptitle("Boxplots of Selected Features", fontsize=16, y=1.02)
    plt.show()

# Run visualizations
plot_histograms(train, cols)
plot_boxplots(train, cols)

### IQR-Based Outlier Capping (Winsorization)

In [ ]:
def cap_outliers_iqr(df, column, factor=1.5):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    df[column] = df[column].clip(lower, upper)

# Columns to treat
cols = ['LotFrontage', 'LotArea', 'MasVnrArea', 'TotalBsmtSF', 
        'GrLivArea']

for col in cols:
    cap_outliers_iqr(train, col)

In [ ]:
# train['SalePrice_Log'] = np.log1p(train['SalePrice'])

### Check outlier capping results

In [ ]:
# Columns of interest
cols = ['LotFrontage', 'LotArea', 'MasVnrArea', 'TotalBsmtSF', 
        'GrLivArea','SalePrice']

# Set plot style
sns.set(style="whitegrid")

# Plot histograms
def plot_histograms(df, columns):
    plt.figure(figsize=(16, 12))
    for i, col in enumerate(columns):
        plt.subplot(4, 2, i + 1)
        sns.histplot(df[col].dropna(), bins=30, kde=True)
        plt.title(f'Distribution of {col}')
    plt.tight_layout()
    plt.suptitle("Histograms of Selected Features", fontsize=16, y=1.02)
    plt.show()

# Plot boxplots
def plot_boxplots(df, columns):
    plt.figure(figsize=(16, 12))
    for i, col in enumerate(columns):
        plt.subplot(4, 2, i + 1)
        sns.boxplot(x=df[col])
        plt.title(f'Boxplot of {col}')
    plt.tight_layout()
    plt.suptitle("Boxplots of Selected Features", fontsize=16, y=1.02)
    plt.show()

# Run visualizations
plot_histograms(train, cols)
plot_boxplots(train, cols)

### **2. Sparse Features**

Several variables are extremely sparse — most observations contain zeros. These include **BsmtFinSF2**, **3SsnPorch**, **ScreenPorch**, **BsmtHalfBath**, and **EnclosedPorch**. Because they appear in a small fraction of homes, they provide limited variance in their raw form. If these features show no strong relationship with the target variable or are too rare, they could potentially be dropped.
> 🟢 **Action**: Check if those features show strong relationship with the target variable or are too rare, they could potentially be dropped. Otherwise, we should consider converting them into binary features indicating presence.

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(train[['BsmtFinSF2', '3SsnPorch', 'ScreenPorch', 'BsmtHalfBath', 'EnclosedPorch','SalePrice']].corr(numeric_only=True), annot=True,cbar=True, cmap='viridis')

In [ ]:
cols_to_drop_corr = [
    'BsmtFinSF2',
    '3SsnPorch',
    'ScreenPorch',
    'BsmtHalfBath',
    'EnclosedPorch'
]

train.drop(columns=cols_to_drop_corr, inplace=True)
print(f"{len(cols_to_drop_corr)} low-correlation columns dropped. New shape: {train.shape}")

In [ ]:
train.columns

In [ ]:
train.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(22,10))
sns.heatmap(data=train.corr(numeric_only=True),annot=True,cmap='viridis')

#### ✅ **Features To Keep**:

These are consistently important across housing models and highly correlated:

* `OverallQual`, `GrLivArea`, `GarageCars`, `GarageArea`
* `TotalBsmtSF`, `1stFlrSF`, `Fireplaces`, `FullBath`
* `YearBuilt`, `MasVnrArea`, `YearRemodAdd`, `TotRmsAbvGrd`
* `LotArea`, `LotFrontage`

#### ❓ **Dropping**:

If your model underperforms or overfits:

* `OverallCond`,`KitchenAbvGr` , `MSSubClass`(weak or negative correlation)
* With attention `BedroomAbvGr`, `BsmtUnfSF`, `BsmtFullBath`, `HalfBath`

#### 🧠 **Feature Engineering Ideas**:

* **Total Bathrooms** = `FullBath` + 0.5 × `HalfBath` + `BsmtFullBath` + 0.5 × `BsmtHalfBath`
* **House Age** = `YrSold` - `YearBuilt`
* **Remodel Age** = `YrSold` - `YearRemodAdd`
---



In [ ]:
cols = ['OverallCond','MSSubClass', 'BedroomAbvGr', 'BsmtUnfSF', 'BsmtFullBath', 'HalfBath']

plt.figure(figsize=(18, 20))

for i, col in enumerate(cols):
    # Countplot (to check imbalance)
    plt.subplot(len(cols), 2, 2*i + 1)
    sns.countplot(data=train, x=col, palette="pastel")
    plt.title(f'Count of {col}')
    plt.xticks(rotation=45)

    # Boxplot vs SalePrice
    plt.subplot(len(cols), 2, 2*i + 2)
    if train[col].nunique() < 20:  # treat as categorical
        sns.boxplot(data=train, x=col, y='SalePrice', palette="muted")
    else:
        sns.scatterplot(data=train, x=col, y='SalePrice', alpha=0.5)
    plt.title(f'SalePrice vs {col}')
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(data=train,x='BsmtUnfSF',y='TotalBsmtSF')

#### Based on the visual analysis of the selected variables, several features appear to have limited or no predictive value for `SalePrice`. `OverallCond`, `BedroomAbvGr`, and `HalfBath` show weak or no correlation with the target and are heavily imbalanced, making them strong candidates for removal. `MSSubClass` displays some variation in price. `BsmtUnfSF`, while continuous, presents a weak positive trend. `BsmtFullBath` shows a slight pattern with `SalePrice`, so it can be optionally retained. Overall, these insights support dropping low-impact features and transforming others to enhance model performance.


In [ ]:
train.drop(columns=['OverallCond', 'BedroomAbvGr', 'HalfBath'], inplace=True)

### **3. Ordinal / Categorical-like Numeric Features**

Some numerical variables actually represent discrete, ordered categories. We will reduce the number of features by grouping semantically similar variables into new, aggregated or engineered features. 
> 🟢 **Action**: These features are best treated as ordinal during modeling.

In [ ]:
train['Basement_TotalSF'] = train['BsmtFinSF1'] + train['BsmtUnfSF']
train['Has_Basement'] = (train['TotalBsmtSF'] > 0).astype(int)
train['Basement_Bath_Indicator'] = (train['BsmtFullBath'] > 0).astype(int)
train['Total_Livable_Area'] = train['1stFlrSF'] + train['2ndFlrSF']
train['Total_Bathrooms'] = train['FullBath'] + train['BsmtFullBath']
train['Is_Kitchen_Excellent'] = (train['KitchenQual'] == 'Ex').astype(int)
train['Has_Fireplace'] = (train['Fireplaces'] > 0).astype(int)
train['Garage_Capacity'] = train['GarageCars'] * train['GarageArea']
train['Has_Garage'] = (train['GarageArea'] > 0).astype(int)
train['Total_Outdoor_SF'] = train['WoodDeckSF'] + train['OpenPorchSF']
train['House_Age'] = train['YrSold'] - train['YearBuilt']
train['Remodel_Age'] = train['YrSold'] - train['YearRemodAdd']

> #### Final Columns

In [ ]:
train.drop(columns=['BsmtFinSF1','BsmtUnfSF','TotalBsmtSF','BsmtFullBath',
'1stFlrSF','2ndFlrSF','FullBath','KitchenQual','Fireplaces',
'GarageCars','GarageArea','WoodDeckSF','OpenPorchSF','YrSold',
'YearBuilt','YearRemodAdd'],inplace=True)

In [ ]:
final_column_features = train.columns
final_column_features

### Visualizations of new features

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Basement Total SF vs SalePrice
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Basement_TotalSF', y='SalePrice', data=train)
plt.title('Basement Total SF vs SalePrice')
plt.show()

# 2. SalePrice by Basement Presence
plt.figure(figsize=(8, 5))
sns.boxplot(x='Has_Basement', y='SalePrice', data=train)
plt.title('SalePrice by Basement Presence')
plt.show()

# 3. SalePrice by Basement Full Bath Presence
plt.figure(figsize=(8, 5))
sns.boxplot(x='Basement_Bath_Indicator', y='SalePrice', data=train)
plt.title('SalePrice by Basement Full Bath Presence')
plt.show()

# 4. Total Livable Area vs SalePrice
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Total_Livable_Area', y='SalePrice', data=train)
plt.title('Total Livable Area vs SalePrice')
plt.show()

# 5. SalePrice by Total Bathrooms
plt.figure(figsize=(8, 5))
sns.boxplot(x='Total_Bathrooms', y='SalePrice', data=train)
plt.title('SalePrice by Total Bathrooms')
plt.show()

# 6. SalePrice by Excellent Kitchen
plt.figure(figsize=(8, 5))
sns.boxplot(x='Is_Kitchen_Excellent', y='SalePrice', data=train)
plt.title('SalePrice by Excellent Kitchen')
plt.show()

# 7. SalePrice by Fireplace Presence
plt.figure(figsize=(8, 5))
sns.boxplot(x='Has_Fireplace', y='SalePrice', data=train)
plt.title('SalePrice by Fireplace Presence')
plt.show()

# 8. Garage Capacity vs SalePrice
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Garage_Capacity', y='SalePrice', data=train)
plt.title('Garage Capacity vs SalePrice')
plt.show()

# 9. SalePrice by Garage Presence
plt.figure(figsize=(8, 5))
sns.boxplot(x='Has_Garage', y='SalePrice', data=train)
plt.title('SalePrice by Garage Presence')
plt.show()

# 10. Total Outdoor SF vs SalePrice
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Total_Outdoor_SF', y='SalePrice', data=train)
plt.title('Total Outdoor SF vs SalePrice')
plt.show()

# 11. House Age vs SalePrice
plt.figure(figsize=(8, 5))
sns.scatterplot(x='House_Age', y='SalePrice', data=train)
plt.title('House Age vs SalePrice')
plt.show()

# 12. Remodel Age vs SalePrice
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Remodel_Age', y='SalePrice', data=train)
plt.title('Remodel Age vs SalePrice')
plt.show()

# Data Preparation

In [ ]:
X = train.drop("SalePrice", axis=1)
y = train["SalePrice"]

In [ ]:
X.shape

In [ ]:
y.shape

## Data Splitting

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12)

In [ ]:
x_train.head()

## Data Transformation 

In [ ]:
# Identify object (categorical) columns in x_train
object_cols = x_train.select_dtypes(include="object").columns

# Limit to first 34 object columns (or define manually if needed)
selected_ordinal_cols = object_cols[:34]

# One-hot encode these columns in x_train
x_train_dummies = pd.get_dummies(x_train[selected_ordinal_cols], drop_first=True)

# One-hot encode same columns in x_test (use same categories as train)
x_test_dummies = pd.get_dummies(x_test[selected_ordinal_cols], drop_first=True)

# Align the train and test sets to have the same columns
x_train_dummies, x_test_dummies = x_train_dummies.align(x_test_dummies, join='left', axis=1, fill_value=0)

# Drop original ordinal columns from x_train and x_test
x_train = x_train.drop(columns=selected_ordinal_cols)
x_test = x_test.drop(columns=selected_ordinal_cols)

# Concatenate dummies to the rest of the features
x_train = pd.concat([x_train, x_train_dummies], axis=1)
x_test = pd.concat([x_test, x_test_dummies], axis=1)

In [ ]:
y_train

# Data Scaling

# x_train and x_test scaled with StandardScaler

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# Initial Modeling & Hypothesis Testing:

In [ ]:
# List of linear regression models to apply
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor()
}

In [ ]:
# Function to evaluate model performance
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [ ]:
# Dictionary to store the results
results = {}

In [ ]:
for name, model in models.items(): # When you call items() on a dictionary, returns a list of the dictionary’s key-value tuple pairs.
                                   # Here "name" represents the "key", and "model" represents the "value"
      model.fit(x_train, y_train)
      y_pred = model.predict(x_test)

        # Compute evaluation metrics
      mae, rmse, r2 = evaluate_model(y_test, y_pred)
      results[name] = {"MAE": mae, "RMSE": rmse, "R²": r2}

In [ ]:
# Convert results to a DataFrame for better visualization
results_train= pd.DataFrame(results).T
print(results_train)

### 🔍 Detailed Interpretation:

| Model                 | MAE ↓            | RMSE ↓                    | R² ↑          | Verdict                                                    |
| --------------------- | ---------------- | ------------------------- | ------------- | ---------------------------------------------------------- |
Ridge                                   |
| **Decision Tree**     | Overfits         | High error                | Moderate      | ⚠️ Simpler but less here                                           |
| **Linear Regression** | Poor             | Poor                      | Low R²        | ❌ 
---

We will use the **GradientBoostingRegressor** 

# Model Generation

In [ ]:
y_pred=model.predict(x_test)
accuracy=r2_score(y_test,y_pred)
print(accuracy)

In [ ]:
import pickle
with open('house_price_prediction.pickle','wb') as f:
    pickle.dump(model,f)

# Prediction

In [ ]:
test = pd.read_csv("test.csv")

In [ ]:
test.drop(columns=['PoolQC','PoolArea'],inplace=True)
test['FireplaceQu'].fillna('None', inplace=True)
test['MiscFeature'].fillna('None', inplace=True)
test['Fence'].fillna('None', inplace=True)
test.drop(columns=['Street','Alley'],inplace=True)
test.loc[(test['MasVnrType'].isnull()) & (test['MasVnrArea'] == 0), 'MasVnrType'] = 'None'
test['MasVnrType'] = test['MasVnrType'].fillna('Unknown')
test['LotFrontage'] = test.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
garage_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
test[garage_cols] = test[garage_cols].fillna('None')
test['GarageYrBlt'] = test['GarageYrBlt'].fillna(test['YearBuilt'])
bsmt_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
test[bsmt_cols] = test[bsmt_cols].fillna('None')
test['MasVnrArea'] = test['MasVnrArea'].fillna(0)
test['Electrical'] = test['Electrical'].fillna(test['Electrical'].mode()[0])

In [ ]:
test['Basement_TotalSF'] = test['BsmtFinSF1'] + test['BsmtUnfSF']
test['Has_Basement'] = (test['TotalBsmtSF'] > 0).astype(int)
test['Basement_Bath_Indicator'] = (test['BsmtFullBath'] > 0).astype(int)
test['Total_Livable_Area'] = test['1stFlrSF'] + test['2ndFlrSF']
test['Total_Bathrooms'] = test['FullBath'] + test['BsmtFullBath']
test['Is_Kitchen_Excellent'] = (test['KitchenQual'] == 'Ex').astype(int)
test['Has_Fireplace'] = (test['Fireplaces'] > 0).astype(int)
test['Garage_Capacity'] = test['GarageCars'] * test['GarageArea']
test['Has_Garage'] = (test['GarageArea'] > 0).astype(int)
test['Total_Outdoor_SF'] = test['WoodDeckSF'] + test['OpenPorchSF']
test['House_Age'] = test['YrSold'] - test['YearBuilt']
test['Remodel_Age'] = test['YrSold'] - test['YearRemodAdd']

In [ ]:
X_test_final = test[[col for col in final_column_features if col != "SalePrice"]]

X_test_final = pd.get_dummies(X_test_final)
X_test_final = X_test_final.reindex(columns=x_train.columns, fill_value=0)  

In [ ]:
X_test_final = X_test_final.fillna(0)

In [ ]:
X_test_final

In [ ]:
y_pred = model.predict(X_test_final)

submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": y_pred
})

submission.to_csv("submission.csv", index=False)
print("✅ Submission file saved as `submission.csv`.")